In [1]:
import pandas as pd

df = pd.read_csv('../data/employee_attrition_raw.csv')  
df = df.dropna(axis=1, how='all')
print('Shape:', df.shape)
df.head()

C:\Users\Hager\AppData\Local\Temp\ipykernel_3280\2285198129.py:3: DtypeWarning: Columns (5,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/employee_attrition_raw.csv')


Shape: (1045, 10)


,Employee_ID,Department,Job_Role,Education_Level,Age,Years_At_Company,Monthly_Income,Performance_Score,Join_Date,Attrition
0,601,Engineering,Software Engineer,Bachelor,21,13.0,7155.17,92.0,7/15/2011,No
1,737,SALES,Sales Executive,master,30,2.0,5122.52,59.0,9/22/2022,No
2,315,Finance,Accountant,Bachelor's,37,0.0,4578.13,80.0,6/18/2024,No
3,816,SALES,Sales Manager,Bachelor,31,0.0,3649.32,NaN,8/15/2024,No
4,979,marketing,Marketing Analyst,Bachelor,42,2.0,4218.21,75.0,9/7/2022,No


Data cleanning 

In [2]:
for C in ['Department', 'Education_Level', 'Job_Role','Attrition']:
    print(f'--- {C} ---')
    print(df[C].value_counts(dropna=False)) # NAN
    print()

--- Department ---
Department
Engineering        182
Sales              167
Marketing          100
Finance             79
HR                  72
SALES               50
sales               49
engineering         46
Engineeering        42
FINANCE             36
Sales               29
Eng.                28
finance             28
Human Resources     28
MARKETING           25
marketing           21
Marketing           18
hr                  16
Finance             15
H.R.                 7
ENGINEERING          6
ENGINEEERING         1
Name: count, dtype: int64

--- Education_Level ---
Education_Level
Bachelor       347
Master         192
High School    123
bachelor        96
Bachelor's      72
PhD             51
Master's        46
master          41
high school     28
HS              22
Ph.D.           15
phd             12
Name: count, dtype: int64

--- Job_Role ---
Job_Role
Account Rep           109
Software Engineer     105
Sales Executive       100
QA Engineer           100
Senior Engin

In [3]:
dept_map = {
    'sales': 'Sales',
    'engineering': 'Engineering', 'eng.': 'Engineering', 'engineeering': 'Engineering',
    'marketing': 'Marketing',
    'finance': 'Finance',
    'hr': 'HR', 'human resources': 'HR', 'h.r.': 'HR',
}

edu_map = {
    'bachelor': 'Bachelor', "bachelor's": 'Bachelor',
    'master': 'Master', "master's": 'Master',
    'high school': 'High School', 'hs': 'High School',
    'phd': 'PhD', 'ph.d.': 'PhD',
}

def standraize(series, mapping):
    normalized = series.astype(str).str.strip().str.lower()
    return normalized.map(mapping).fillna(series)

df['Department'] = standraize(df['Department'], dept_map)
df['Education_Level'] = standraize(df['Education_Level'], edu_map)

print(df['Department'].value_counts())
print()
print(df['Education_Level'].value_counts())

Department
Engineering    305
Sales          295
Marketing      164
Finance        158
HR             123
Name: count, dtype: int64

Education_Level
Bachelor       515
Master         279
High School    173
PhD             78
Name: count, dtype: int64


In [ ]:
# Ensure each column has the correct type
# errors='coerce' converts any value that can't be parsed into NaN
df['Employee_ID'] = df['Employee_ID'].astype('Int64')
df['Age'] = df['Age'].astype('Int64')
df['Years_At_Company'] = pd.to_numeric(df['Years_At_Company'],errors='coerce').astype('Int64')

df['Monthly_Income'] = pd.to_numeric(
    df['Monthly_Income'],
    errors='coerce'
)

df['Performance_Score'] = pd.to_numeric(
    df['Performance_Score'],
    errors='coerce'
)

df['Join_Date'] = pd.to_datetime(
    df['Join_Date'],
    errors='coerce'
)

Drop fully duplicated rows
separately check for duplicate Employee_ID values alone (even if other columns differ), since that points to a deeper data-quality issue that needs manual review

In [5]:
duplicated_row = df[df.duplicated(keep=False)]
# print('duplicated_row', len(duplicated_row))

df = df.drop_duplicates(keep='first')
print('after removing duplicates:', df.shape)

id_duplicated= df[df.duplicated(subset='Employee_ID', keep=False)].sort_values('Employee_ID')
print('Employee_ID_Duplicated:', len(id_duplicated))


after removing duplicates: (1000, 10)
Employee_ID_Duplicated: 0


Recheck unique values and summary stats after initial cleaning   
to find hiding missing values before handling them

In [6]:
print("=== Categorical Unique Values ===")
for col in ['Department', 'Job_Role', 'Education_Level', 'Attrition']:
    if col in df.columns:
        print(f"--- {col} ---")
        print(df[col].unique())
        print()

print("=== Numerical Summary ===")
print(df[['Age', 'Years_At_Company', 'Monthly_Income', 'Performance_Score']].describe())


print("\n=== Direct Missing Values (NaNs) ===")
print(df.isnull().sum())

=== Categorical Unique Values ===
--- Department ---
['Engineering' 'Sales' 'Finance' 'Marketing' 'HR']

--- Job_Role ---
['Software Engineer' 'Sales Executive' 'Accountant' 'Sales Manager'
 'Marketing Analyst' 'Account Rep' 'Marketing Manager' 'Finance Manager'
 'Senior Engineer' 'Content Specialist' 'HR Specialist' 'QA Engineer'
 'Recruiter' 'Financial Analyst' 'HR Manager']

--- Education_Level ---
['Bachelor' 'Master' 'High School' 'PhD']

--- Attrition ---
['No' 'Yes']

=== Numerical Summary ===
             Age  Years_At_Company  Monthly_Income  Performance_Score
count     1000.0             956.0      943.000000         960.000000
mean      34.495         -8.258368     5490.924804          71.247917
std    10.500964        107.010217     5961.425990          12.080759
min          4.0            -999.0     -999.000000          31.000000
25%         29.0               1.0     3795.475000          63.000000
50%         34.0               2.0     4749.140000          71.000000
75% 

Missing Values

The value -999 is commonly used as a placeholder for "missing" in some systems instead of a true null, so it needs to be converted to NaN to be handled properly. Similarly, ages outside a realistic range (under 20 or over 80) are treated as data-entry errors and set to NaN, to be imputed in the next step.

In [ ]:
import numpy as np

df['Years_At_Company'] = df['Years_At_Company'].replace(-999, np.nan)
df['Monthly_Income'] = df['Monthly_Income'].replace(-999, np.nan)


df.loc[(df['Age'] < 20 ) | (df['Age'] > 80), 'Age'] = np.nan

print(" Missing_values")
print(df[['Age', 'Years_At_Company', 'Monthly_Income', 'Performance_Score']].isnull().sum())

 Missing_values
Age                  10
Years_At_Company     55
Monthly_Income       70
Performance_Score    40
dtype: int64


Impute missing values with KNN

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

num_cols = ['Age', 'Years_At_Company', 'Monthly_Income', 'Performance_Score']

# scalling for large number , it relies on Euclidean distance
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df[num_cols]), columns=num_cols)

imputer = KNNImputer(n_neighbors=5, weights='uniform', metric='nan_euclidean')
df_imputed_scaled = imputer.fit_transform(df_scaled)

# inverse scalling
df[num_cols] = scaler.inverse_transform(df_imputed_scaled)


In [9]:
df['Age'] = df['Age'].round().astype('Int64')
df['Years_At_Company'] = df['Years_At_Company'].round().astype('Int64')

Removing Outliers

In [10]:
from sklearn.neighbors import LocalOutlierFactor
import numpy as np
lof = LocalOutlierFactor()

num_cols = ['Age', 'Years_At_Company', 'Monthly_Income', 'Performance_Score']
OutliersDetectionForNum = df[num_cols].copy()
OutliersDetectionForNum['LOF'] = lof.fit_predict(OutliersDetectionForNum)
#print(OutliersDetectionForNum)

Outliers_indices = np.where(OutliersDetectionForNum['LOF'] == -1)[0]
# Resetting the index of the dataframe before removing outliers
df.reset_index(inplace=True, drop=True)
# Removing the outliers
df.drop(Outliers_indices, inplace=True)
print(df.shape)

(970, 10)


Encoding

Education_Level uses ordinal encoding because it's a naturally ordered category (High School < Bachelor < Master < PhD), and the model benefits from preserving that order.  

Attrition is mapped to 0/1 since it's binary.  

Department and Job_Role have no inherent order, so one-hot encoding (get_dummies) is used instead of numeric encoding

In [11]:
education_order = {
    'High School': 1,
    'Bachelor': 2,
    'Master': 3,
    'PhD': 4
}
df['Education_Level'] = df['Education_Level'].map(education_order)

df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

df = pd.get_dummies(df, columns=['Department', 'Job_Role'], drop_first=True, dtype=int)

print("After Encoding",df.shape)
df.info()

After Encoding (970, 26)
<class 'pandas.core.frame.DataFrame'>
Index: 970 entries, 0 to 998
Data columns (total 26 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Employee_ID                  970 non-null    Int64         
 1   Education_Level              970 non-null    int64         
 2   Age                          970 non-null    Int64         
 3   Years_At_Company             970 non-null    Int64         
 4   Monthly_Income               970 non-null    float64       
 5   Performance_Score            970 non-null    float64       
 6   Join_Date                    970 non-null    datetime64[ns]
 7   Attrition                    970 non-null    int64         
 8   Department_Finance           970 non-null    int64         
 9   Department_HR                970 non-null    int64         
 10  Department_Marketing         970 non-null    int64         
 11  Department_Sales         

class Imbalanc

check number before imbalanc

In [12]:
print(df['Attrition'].value_counts())
print(df['Attrition'].value_counts(normalize=True)*100)

Attrition
0    814
1    156
Name: count, dtype: int64
Attrition
0    83.917526
1    16.082474
Name: proportion, dtype: float64


In [13]:
df.to_csv('../data/employee_attrition_clean.csv', index=False)
print('done:', df.shape)

done: (970, 26)
